# Assignment: Video Quality Inference

To this point in the class, you have learned various techniques for leading and analyzing packet captures of various types, generating features from those packet captures, and training and evaluating models using those features.

In this assignment, you will put all of this together, using a network traffic trace to train a model to automatically infer video quality of experience from a labeled traffic trace.

## Part 1: Warmup

The first part of this assignment builds directly on the hands-on activities but extends them slightly.

### Extract Features from the Network Traffic

Load the `netflix.pcap` file, which is a packet trace that includes network traffic. 


In [1]:
import logging
logging.getLogger("scapy.runtime").setLevel(logging.ERROR)

from netml.pparser.parser import PCAP
from netml.utils.tool import dump_data, load_data
import pandas as pd

# Load the netflix pcap and require at least 2 packets per flow
pcap = PCAP("../notebooks/data/netflix.pcap", flow_ptks_thres=2)
pcap.pcap2flows()

print(f"Number of flows: {len(pcap.flows)}")
print(f"Computed split interval: {pcap.interval:.4f} seconds")

Number of flows: 184
Computed split interval: 277.8675 seconds


In [2]:
# Look at the first few flows
for fid, pkts in pcap.flows[:5]:
    print(f"Flow: {fid}")
    print(f"  Packets: {len(pkts)}")
    print(f"  First packet: {pkts[0].summary()}")
    print()

Flow: ('192.168.43.72', '172.217.18.195', 58443, 443, 6)
  Packets: 12
  First packet: Ether / IP / TCP 192.168.43.72:58443 > 172.217.18.195:https S

Flow: ('192.168.43.72', '216.58.209.228', 58444, 443, 6)
  Packets: 67
  First packet: Ether / IP / TCP 192.168.43.72:58444 > 216.58.209.228:https S

Flow: ('192.168.43.72', '216.58.209.228', 58445, 443, 6)
  Packets: 5
  First packet: Ether / IP / TCP 192.168.43.72:58445 > 216.58.209.228:https S

Flow: ('192.168.43.72', '172.217.18.195', 58446, 443, 6)
  Packets: 5
  First packet: Ether / IP / TCP 192.168.43.72:58446 > 172.217.18.195:https S

Flow: ('192.168.43.72', '172.217.18.195', 58447, 443, 6)
  Packets: 29
  First packet: Ether / IP / TCP 192.168.43.72:58447 > 172.217.18.195:https S



### Identifying the Service Type

Use the DNS traffic to filter the packet trace for Netflix traffic.

In [3]:
from scapy.all import rdpcap, DNS, DNSRR

NF_DOMAINS = ["nflxvideo", "netflix", "nflxso", "nflxext"]

packets = rdpcap("../notebooks/data/netflix.pcap")

# Find Netflix IPs from DNS responses
netflix_ips = set()
for pkt in packets:
    if pkt.haslayer(DNS) and pkt[DNS].ancount > 0: # when pkt[DNS].ancount == 0, this is a lookup query for IP address
        for i in range(pkt[DNS].ancount):
            rr = pkt[DNS].an if i == 0 else pkt[DNS].an[i]
            if hasattr(rr, 'rrname') and hasattr(rr, 'rdata'):
                name = rr.rrname.decode() if isinstance(rr.rrname, bytes) else rr.rrname
                if any(d in name.lower() for d in NF_DOMAINS) and rr.type == 1: # if domain contains any of the key words, then it's from netflix -> add to set
                    netflix_ips.add(rr.rdata)

print(f"Found {len(netflix_ips)} Netflix IP addresses:")
for ip in sorted(netflix_ips):
    print(f"  {ip}")

# Filter flows to only those involving a Netflix IP
netflix_flows = [(fid, pkts) for fid, pkts in pcap.flows
                 if fid[0] in netflix_ips or fid[1] in netflix_ips]

print(f"\nFiltered from {len(pcap.flows)} total flows to {len(netflix_flows)} Netflix flows")

Found 15 Netflix IP addresses:
  198.38.120.130
  198.38.120.134
  198.38.120.137
  198.38.120.153
  198.38.120.162
  198.38.120.164
  198.38.120.166
  198.38.120.167
  34.252.77.54
  52.19.39.146
  52.208.128.101
  52.210.133.255
  52.210.19.176
  52.48.148.78
  52.48.8.150

Filtered from 184 total flows to 110 Netflix flows


### Generate Statistics

Generate statistics and features for the Netflix traffic flows. Use the `netml` library or any other technique that you choose to generate a set of features that you think would be good features for your model. 

In [4]:
import copy

# Create a copy so we don't modify the original pcap
netflix_pcap = copy.copy(pcap)
netflix_pcap.flows = netflix_flows

# Extract STATS features (summary statistics per flow)
netflix_pcap.flow2features("STATS", fft=False, header=False)

print(f"Feature matrix shape: {netflix_pcap.features.shape}")
print(f"(rows = Netflix flows, columns = statistical features)")

Feature matrix shape: (110, 12)
(rows = Netflix flows, columns = statistical features)


In [9]:
feature_names = ["duration", "pkts_per_sec", "bytes_per_sec", 
                 "mean_pkt_size", "std_pkt_size", "median_pkt_size",
                 "q1_pkt_size", "q3_pkt_size", "min_pkt_size", 
                 "max_pkt_size", "num_pkts", "total_bytes"]

stats_df = pd.DataFrame(netflix_pcap.features, columns=feature_names)
stats_df.head(50)

,duration,pkts_per_sec,bytes_per_sec,mean_pkt_size,std_pkt_size,median_pkt_size,q1_pkt_size,q3_pkt_size,min_pkt_size,max_pkt_size,num_pkts,total_bytes
0,14.440177,0.831015,73.683308,88.666667,48.140997,66.0,66.0,69.0,66.0,200.0,12.0,1064.0
1,14.442865,0.761622,69.099863,90.727273,49.772374,66.0,66.0,72.0,66.0,200.0,11.0,998.0
2,14.437420,0.831173,73.697377,88.666667,48.140997,66.0,66.0,69.0,66.0,200.0,12.0,1064.0
3,138.854511,1.497971,184.560082,123.206731,66.443951,66.0,66.0,200.0,54.0,200.0,208.0,25627.0
4,74.817607,0.360878,45.203263,125.259259,66.350692,66.0,66.0,200.0,54.0,200.0,27.0,3382.0
5,17.894865,0.726465,63.817190,87.846154,46.815261,66.0,66.0,78.0,54.0,200.0,13.0,1142.0
6,430.839378,0.510631,80.621693,157.886364,62.623919,66.0,200.0,200.0,54.0,200.0,220.0,34735.0
7,17.891358,0.558929,50.750759,90.800000,53.022259,66.0,66.0,75.0,54.0,200.0,10.0,908.0
8,17.886625,0.614985,54.454097,88.545455,51.054986,66.0,66.0,72.0,54.0,200.0,11.0,974.0
9,70.155289,0.228065,24.488531,107.375000,61.541526,66.0,66.0,194.0,54.0,200.0,16.0,1718.0


I chose STATS since the main thing that determines video quality is how much data is being sent per second which can be seen with features like bytes/sec and packets/sec. I also considered time-series approaches like SAMP or IAT, but STATS always gives a fixed 12 features per flow no matter how long it lasts, which keeps things simple and avoids issues with padding or truncating flows to the same length.

### Inferring Segment downloads

In addition to the features that you could generate using the `netml` library or similar, add to your feature vector a "segment downloads rate" feature, which indicates the number of video segments downloaded for a given time window.

Note: If you are using the `netml` library, generating features with `SAMP` style options may be useful, as this option gives you time windows, and you can then simply add the segment download rate to that existing dataframe.

In [11]:
import numpy as np

def count_segments(pkts):
    """Count video segments by detecting bursts separated by zero-payload packets."""
    segments = 0
    in_burst = False
    for pkt in pkts:
        payload_len = len(pkt) - 54
        if payload_len > 0:
            if not in_burst:
                segments += 1
                in_burst = True
        else:
            in_burst = False
    return segments

# Compute segment count and rate for each Netflix flow
seg_counts = []
seg_rates = []
for fid, pkts in netflix_flows:
    seg_count = count_segments(pkts)
    duration = float(pkts[-1].time - pkts[0].time)
    rate = seg_count / duration if duration > 0 else 0
    seg_counts.append(seg_count)
    seg_rates.append(rate)

# Add as new columns to our feature DataFrame
stats_df["seg_count"] = seg_counts
stats_df["seg_download_rate"] = seg_rates
print(f"Added segment count and download rate features")
stats_df[["duration", "bytes_per_sec", "seg_count", "seg_download_rate"]].head(10)

Added segment count and download rate features


,duration,bytes_per_sec,seg_count,seg_download_rate
0,14.440177,73.683308,1,0.069251
1,14.442865,69.099863,1,0.069238
2,14.437420,73.697377,1,0.069264
3,138.854511,184.560082,2,0.014404
4,74.817607,45.203263,2,0.026732
5,17.894865,63.817190,1,0.055882
6,430.839378,80.621693,5,0.011605
7,17.891358,50.750759,1,0.055893
8,17.886625,54.454097,1,0.055908
9,70.155289,24.488531,2,0.028508


## Part 2: Video Quality Inference

You will now load the complete video dataset from a previous study to train and test models based on these features to automatically infer the quality of a streaming video flow.

For this part of the assignment, you will need two pickle files, which we provide for you by running the code below:

```

!gdown 'https://drive.google.com/uc?id=1N-Cf4dJ3fpak_AWgO05Fopq_XPYLVqdS' -O netflix_session.pkl
!gdown 'https://drive.google.com/uc?id=1PHvEID7My6VZXZveCpQYy3lMo9RvMNTI' -O video_dataset.pkl

```

### Load the File

Load the video dataset pickle file.

### Clean the File

1. The dataset contains video resolutions that are not valid. Remove entries in the dataset that do not contain a valid video resolution. Valid resolutions are 240, 360, 480, 720, 1080.

2. The file also contains columns that are unnecessary (in fact, unhelpful!) for performing predictions. Identify those columns, and remove them.

**Briefly explain why you removed those columns.**

### Prepare Your Data

Prepare your data matrix, determine your features and labels, and perform a train-test split on your data.

### Train and Tune Your Model

1. Select a model of your choice.
2. Train the model using your training data.

### Tune Your Model

Perform hyperparameter tuning to find optimal parameters for your model.

### Evaluate Your Model

Evaluate your model accuracy according to the following metrics:

1. Accuracy
2. F1 Score
3. Confusion Matrix
4. ROC/AUC

## Part 3: Predict the Ongoing Resolution of a Real Netflix Session

Now that you have your model, it's time to put it in practice!

Use a preprocessed Netflix video session to infer **and plot** the resolution at 10-second time intervals.